# Generative Adversarial Networks (GANs)

---

## 1. Introduction to GANs

**Generative Adversarial Networks (GANs)** were introduced by Ian Goodfellow in 2014 and revolutionized the field of generative modeling. The core idea is brilliantly simple yet powerful: **two neural networks competing against each other**.

### The Analogy: Art Forger vs Detective

Imagine this scenario:
- **The Art Forger (Generator)**: Creates fake paintings, trying to make them look authentic
- **The Art Detective (Discriminator)**: Examines paintings to determine if they're real or fake

**The Game:**
1. The Forger creates a fake painting
2. The Detective tries to identify it as fake
3. The Forger learns from the Detective's feedback
4. Both get better over time
5. Eventually, the Forger becomes so good that the Detective can't tell the difference!

This is exactly how GANs work with neural networks.

In [1]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from torch.utils.data import DataLoader
from typing import Dict, Tuple, Optional
import logging
import requests
from PIL import Image
from io import BytesIO
import torchvision.models as models

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set device and random seed for reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

logger.info(f"Using device: {device}")
logger.info("Libraries imported successfully")


## 2. The Adversarial Game Theory

### Mathematical Foundation

GANs solve a **minimax game** between two players:

**Objective Function:**
```
min_G max_D V(D,G) = E[log D(x)] + E[log(1 - D(G(z)))]
```

Where:
- **G**: Generator network
- **D**: Discriminator network
- **x**: Real data samples
- **z**: Random noise (latent vector)
- **G(z)**: Generated (fake) samples
- **D(x)**: Discriminator's prediction for real data
- **D(G(z))**: Discriminator's prediction for fake data

### Training Process

1. **Discriminator Training**: Maximize ability to distinguish real from fake
   - Real images → label = 1
   - Fake images → label = 0

2. **Generator Training**: Minimize discriminator's ability to detect fakes
   - Generate fake images
   - Try to fool discriminator (want discriminator to output 1)

### Nash Equilibrium

The training converges when both networks reach a **Nash Equilibrium**:
- Discriminator outputs 0.5 for all samples (can't distinguish real from fake)
- Generator produces samples indistinguishable from real data


## 3. DCGAN Architecture

**Deep Convolutional GAN (DCGAN)** introduced by Radford et al. in 2015, brought stability to GAN training through architectural innovations.

### Key DCGAN Principles:

1. **Replace pooling layers with strided convolutions** (discriminator) and fractional-strided convolutions (generator)
2. **Use batch normalization** in both generator and discriminator
3. **Remove fully connected hidden layers** for deeper architectures
4. **Use ReLU activation** in generator for all layers except output (Tanh)
5. **Use LeakyReLU activation** in discriminator for all layers

### Why These Choices Matter:

- **Strided Convolutions**: Learn their own spatial downsampling
- **Batch Normalization**: Stabilizes learning, prevents mode collapse
- **ReLU vs LeakyReLU**: ReLU for generation, LeakyReLU to prevent sparse gradients
- **Tanh Output**: Maps to [-1, 1] range, works well with normalized input data


In [2]:
# DCGAN Generator - The "Forger"
class DCGANGenerator(nn.Module):
    """
    DCGAN Generator that creates images from random noise.

    Architecture:
    - Takes random noise (latent vector) as input
    - Uses transposed convolutions to upsample
    - Outputs RGB images
    """

    def __init__(self, latent_dim: int = 100, num_channels: int = 3, feature_maps: int = 64):
        """
        Initialize the Generator.

        Args:
            latent_dim (int): Size of the random noise vector
            num_channels (int): Number of output channels (3 for RGB)
            feature_maps (int): Number of feature maps in generator
        """
        super(DCGANGenerator, self).__init__()

        self.latent_dim = latent_dim

        # Main generator architecture
        self.main = nn.Sequential(
            # Input: latent_dim x 1 x 1
            # First layer: expand the latent vector
            nn.ConvTranspose2d(latent_dim, feature_maps * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_maps * 8),
            nn.ReLU(True),
            # Output: (feature_maps*8) x 4 x 4

            # Second layer: upsample
            nn.ConvTranspose2d(feature_maps * 8, feature_maps * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 4),
            nn.ReLU(True),
            # Output: (feature_maps*4) x 8 x 8

            # Third layer: upsample
            nn.ConvTranspose2d(feature_maps * 4, feature_maps * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 2),
            nn.ReLU(True),
            # Output: (feature_maps*2) x 16 x 16

            # Fourth layer: upsample
            nn.ConvTranspose2d(feature_maps * 2, feature_maps, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps),
            nn.ReLU(True),
            # Output: feature_maps x 32 x 32

            # Final layer: create RGB image
            nn.ConvTranspose2d(feature_maps, num_channels, 4, 2, 1, bias=False),
            nn.Tanh()  # Output values between -1 and 1
            # Output: num_channels x 64 x 64
        )

    def forward(self, noise: torch.Tensor) -> torch.Tensor:
        """
        Generate images from noise.

        Args:
            noise (torch.Tensor): Random noise tensor

        Returns:
            torch.Tensor: Generated images
        """
        return self.main(noise)

# Test the generator
generator = DCGANGenerator().to(device)
test_noise = torch.randn(4, 100, 1, 1).to(device)  # Batch of 4 noise vectors
fake_images = generator(test_noise)
f"Generator output shape: {fake_images.shape}"

'Generator output shape: torch.Size([4, 3, 64, 64])'

In [3]:
# DCGAN Discriminator - The "Detective"
class DCGANDiscriminator(nn.Module):
    """
    DCGAN Discriminator that classifies images as real or fake.

    Architecture:
    - Takes RGB images as input
    - Uses convolutions to downsample
    - Outputs a single probability (real vs fake)
    """

    def __init__(self, num_channels: int = 3, feature_maps: int = 64):
        """
        Initialize the Discriminator.

        Args:
            num_channels (int): Number of input channels (3 for RGB)
            feature_maps (int): Number of feature maps in discriminator
        """
        super(DCGANDiscriminator, self).__init__()

        self.main = nn.Sequential(
            # Input: num_channels x 64 x 64
            # First layer: downsample
            nn.Conv2d(num_channels, feature_maps, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # Output: feature_maps x 32 x 32

            # Second layer: downsample
            nn.Conv2d(feature_maps, feature_maps * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # Output: (feature_maps*2) x 16 x 16

            # Third layer: downsample
            nn.Conv2d(feature_maps * 2, feature_maps * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # Output: (feature_maps*4) x 8 x 8

            # Fourth layer: downsample
            nn.Conv2d(feature_maps * 4, feature_maps * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # Output: (feature_maps*8) x 4 x 4

            # Final layer: classify
            nn.Conv2d(feature_maps * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()  # Output probability between 0 and 1
            # Output: 1 x 1 x 1
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Classify images as real or fake.

        Args:
            images (torch.Tensor): Input images

        Returns:
            torch.Tensor: Probability that images are real
        """
        return self.main(images).view(-1, 1).squeeze(1)

# Test the discriminator
discriminator = DCGANDiscriminator().to(device)
test_images = torch.randn(4, 3, 64, 64).to(device)  # Batch of 4 images
predictions = discriminator(test_images)
f"Discriminator output shape: {predictions.shape}, values: {predictions.detach().cpu().numpy()}"

'Discriminator output shape: torch.Size([4]), values: [0.56797016 0.33470803 0.59432834 0.36887088]'